# Env setup

Create a Conda enviroment:

```
mamba create -n spyce python=3.9

mamba activate spyce

mamba install -c conda-forge pandas numpy scipy matplotlib biopython pybedtools logomaker pysam gcc_linux-64 gxx_linux-64 libstdcxx-ng

pip install spyceATAC
```

To force Conda's `libstdc++.so.6` to take priority over system one, prepend the Conda environment's `lib` directory to your `LD_LIBRARY_PATH`. You can make this permanent when activating the environment, create or edit the file: `$CONDA_PREFIX/etc/conda/activate.d/env_vars.sh` with the following:

```
#!/bin/sh
export LD_LIBRARY_PATH=$CONDA_PREFIX/lib:$LD_LIBRARY_PATH
```

# Input data

sPYce requires as data input the following files:

- a peak matrix per sample and species, rows representing cells, columns representing peaks
- a bed file with peaks in the same order as the peaks in the peak matrix. You need to have either one per peak matrix or one per species
- a reference genome as fasta .fa file per species

### Prepare Mouse data
Download TFIDF normalized data

In [ ]:
wget http://krishna.gs.washington.edu/content/members/ajh24/mouse_atlas_data_release/matrices/atac_matrix.tfidf.qc_filtered.mtx.gz
wget http://krishna.gs.washington.edu/content/members/ajh24/mouse_atlas_data_release/matrices/atac_matrix.tfidf.qc_filtered.peaks.txt
wget http://krishna.gs.washington.edu/content/members/ajh24/mouse_atlas_data_release/matrices/atac_matrix.tfidf.qc_filtered.cells.txt

Load in python

In [ ]:
import anndata as ad
import pandas as pd
from scipy.io import mmread

# Paths
data_dir = "/home/anamaria/cluster/aelek/proj/scATAC_nvec_v2/cross-species/mmus/adult/matrix/"
input_dir = "/home/anamaria/cluster/aelek/proj/scATAC_nvec_v2/results/sPYce/inputs/mmus"

# Load matrix
X = mmread(f"{data_dir}/atac_matrix.tfidf.qc_filtered.mtx.gz").tocsr()

# Load row (peak) and column (cell) names, force string type
var = pd.read_csv(f"{data_dir}/atac_matrix.tfidf.qc_filtered.peaks.txt", header=None)[0].astype(str)
obs = pd.read_csv(f"{data_dir}/atac_matrix.tfidf.qc_filtered.cells.txt", header=None)[0].astype(str)

# Create DataFrames with string index
obs_df = pd.DataFrame(index=obs)
var_df = pd.DataFrame(index=var)

# Create AnnData object
obs_df.index.name = "cell"
var_df.index.name = "peak"
adata = ad.AnnData(X=X.T, obs=obs_df, var=var_df)

# Save as h5ad
adata.write(f"{input_dir}/matrix.h5ad")

# Save peaks as bed
peaks_bed = pd.DataFrame({
    'chrom': var.str.split('_', expand=True)[0],
    'start': var.str.split('_', expand=True)[1].astype(int),
    'end': var.str.split('_', expand=True)[2].astype(int),
    'name': var,
    'score': 0,
    'strand': '.'
})
peaks_bed.to_csv(f"{input_dir}/peaks.bed", sep='\t', header=False, index=False)

### Prepare Nematostella data

Load R data

In [ ]:
library(data.table)

# Paths
data_dir = "/home/anamaria/cluster/aelek/proj/scATAC_nvec_v2/Nematostella_scATAC/results/Peaks/"

# Load matrix
mat_fn = file.path(data_dir, "Matrix-Adult-Gastrula-Peaks-mapped-qnormalized.rds")
X = readRDS(mat_fn)

# Load peaks
peaks_fn = file.path(data_dir, "Peaks_cell_type_mapped.bed")
peaks = fread(peaks_fn, header = FALSE)
setnames(peaks, c("seqnames", "start", "end", "peak", "score", "strand"))

# Peaks in intersect
pks <- intersect(rownames(X), peaks$peak)
length(pks)

# Subset matrix
X = X[pks, ]

# Subset peaks
peaks = peaks[peak %in% pks, ]
peaks = peaks[match(pks, peak), ]

# Rename peaks
peaks[, peak := sprintf("%s:%d-%d", seqnames, start, end)]
rownames(X) = peaks$peak

# Order peaks
setorder(peaks, seqnames, start, end)
X = X[peaks$peak, ]

# Save data
dir.create(file.path(data_dir, "matrix_normalized"), showWarnings = FALSE, recursive = TRUE)
Matrix::writeMM(X, file = file.path(data_dir, "matrix_normalized", "Matrix-Adult-Gastrula-Peaks-mapped-qnormalized.mtx"))
write.table(
    rownames(X),
    file = file.path(data_dir, "matrix_normalized", "peaks.tsv"),
    quote = FALSE, row.names = FALSE, col.names = FALSE
)
write.table(
    colnames(X),
    file = file.path(data_dir, "matrix_normalized", "barcodes.tsv"),
    quote = FALSE, row.names = FALSE, col.names = FALSE
)
fwrite(
    peaks, 
    file = file.path(data_dir, "matrix_normalized", "peaks.bed"), 
    sep = "\t", col.names = FALSE
)

Load in Python

In [ ]:
import anndata as ad
import pandas as pd
from scipy.io import mmread

# Paths
data_dir = "/home/anamaria/cluster/aelek/proj/scATAC_nvec_v2/Nematostella_scATAC/results/Peaks/matrix_normalized"

# Load matrix
X = mmread(f"{data_dir}/Matrix-Adult-Gastrula-Peaks-mapped-qnormalized.mtx").tocsr()

# Load row (peak) and column (cell) names, force string type
var = pd.read_csv(f"{data_dir}/peaks.tsv", header=None)[0].astype(str)
obs = pd.read_csv(f"{data_dir}/barcodes.tsv", header=None)[0].astype(str)

# Create DataFrames with string index
obs_df = pd.DataFrame(index=obs)
var_df = pd.DataFrame(index=var)

# Create AnnData object
obs_df.index.name = "cell"
var_df.index.name = "peak"
adata = ad.AnnData(X=X.T, obs=obs_df, var=var_df)

# Save as h5ad
adata.write(f"{data_dir}/Matrix-Adult-Gastrula-Peaks-mapped-qnormalized.h5ad")


### Prepare *Nematostella* gastrula data

Load R data

In [ ]:
library(data.table)

# Paths
data_dir = "/home/anamaria/cluster/aelek/proj/scATAC_nvec_gastrula/clustering/ArchR/ArchRProj_Nvec_gastrula/PeakMatrix"

# Load matrix
mat_fn = file.path(data_dir, "Matrix-Peaks.rds")
X = readRDS(mat_fn)

# Load peaks
peaks_fn = file.path(data_dir, "peaks.rds")
peaks = as.data.table(readRDS(peaks_fn))
colnames = c("seqnames", "start", "end", "name", "score", "strand")
peaks = peaks[, ..colnames]

# Peaks in intersect
pks <- intersect(rownames(X), peaks$name)
length(pks)

# Subset matrix
X = X[pks, ]

# Subset peaks
peaks = peaks[name %in% pks, ]
peaks = peaks[match(pks, name), ]

# Rename peaks
peaks[start<0, start := 0]
peaks[, name := sprintf("%s:%d-%d", seqnames, start, end)]
rownames(X) = peaks$name

# Save data
input_dir = file.path("results", "sPYce", "inputs", "nvec_gastrula")
dir.create(input_dir, showWarnings = FALSE, recursive = TRUE)
Matrix::writeMM(X, file = file.path(input_dir, "matrix.mtx"))
write.table(
    rownames(X),
    file = file.path(input_dir, "peaks.tsv"),
    quote = FALSE, row.names = FALSE, col.names = FALSE
)
write.table(
    colnames(X),
    file = file.path(input_dir, "barcodes.tsv"),
    quote = FALSE, row.names = FALSE, col.names = FALSE
)
fwrite(
    peaks, 
    file = file.path(input_dir, "peaks.bed"), 
    sep = "\t", col.names = FALSE
)

Load in Pythhon

In [ ]:
import anndata as ad
import pandas as pd
from scipy.io import mmread

# Paths
data_dir = "/home/anamaria/cluster/aelek/proj/scATAC_nvec_v2/Nematostella_scATAC/results/sPYce/inputs/nvec_gastrula/"

# Load matrix
X = mmread(f"{data_dir}/matrix.mtx").tocsr()

# Load row (peak) and column (cell) names, force string type
var = pd.read_csv(f"{data_dir}/peaks.tsv", header=None)[0].astype(str)
obs = pd.read_csv(f"{data_dir}/barcodes.tsv", header=None)[0].astype(str)

# Create DataFrames with string index
obs_df = pd.DataFrame(index=obs)
var_df = pd.DataFrame(index=var)

# Create AnnData object
obs_df.index.name = "cell"
var_df.index.name = "peak"
adata = ad.AnnData(X=X.T, obs=obs_df, var=var_df)

# Save as h5ad
adata.write(f"{data_dir}/matrix.h5ad")


### Prepare *Nematostella* adult data

Load R data

In [ ]:
library(data.table)

# Paths
data_dir = "/home/anamaria/cluster/aelek/proj/scATAC_nvec_v2/clustering/ArchR/ArchRProj_Nvec_TSS4_frag200/PeakMatrix"

# Load matrix
mat_fn = file.path(data_dir, "Matrix-Peaks.rds")
X = readRDS(mat_fn)

# Load peaks
peaks_fn = file.path(data_dir, "peaks.rds")
peaks = as.data.table(readRDS(peaks_fn))
colnames = c("seqnames", "start", "end", "name", "score", "strand")
peaks = peaks[, ..colnames]

# Peaks in intersect
pks <- intersect(rownames(X), peaks$name)
length(pks)

# Subset matrix
X = X[pks, ]

# Subset peaks
peaks = peaks[name %in% pks, ]
peaks = peaks[match(pks, name), ]

# Rename peaks
peaks[start<0, start := 0]
peaks[, name := sprintf("%s:%d-%d", seqnames, start, end)]
rownames(X) = peaks$name

# Save data
input_dir = file.path("results", "sPYce", "inputs", "nvec_adult")
dir.create(input_dir, showWarnings = FALSE, recursive = TRUE)
Matrix::writeMM(X, file = file.path(input_dir, "matrix.mtx"))
write.table(
    rownames(X),
    file = file.path(input_dir, "peaks.tsv"),
    quote = FALSE, row.names = FALSE, col.names = FALSE
)
write.table(
    colnames(X),
    file = file.path(input_dir, "barcodes.tsv"),
    quote = FALSE, row.names = FALSE, col.names = FALSE
)
fwrite(
    peaks, 
    file = file.path(input_dir, "peaks.bed"), 
    sep = "\t", col.names = FALSE
)

Load in Python

In [ ]:
import anndata as ad
import pandas as pd
from scipy.io import mmread

# Paths
data_dir = "/home/anamaria/cluster/aelek/proj/scATAC_nvec_v2/Nematostella_scATAC/results/sPYce/inputs/nvec_adult/"

# Load matrix
X = mmread(f"{data_dir}/matrix.mtx").tocsr()

# Load row (peak) and column (cell) names, force string type
var = pd.read_csv(f"{data_dir}/peaks.tsv", header=None)[0].astype(str)
obs = pd.read_csv(f"{data_dir}/barcodes.tsv", header=None)[0].astype(str)

# Create DataFrames with string index
obs_df = pd.DataFrame(index=obs)
var_df = pd.DataFrame(index=var)

# Create AnnData object
obs_df.index.name = "cell"
var_df.index.name = "peak"
adata = ad.AnnData(X=X.T, obs=obs_df, var=var_df)

# Save as h5ad
adata.write(f"{data_dir}/matrix.h5ad")


# Create KMerCollection

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import polars as pl
from pathlib import Path

First collect paths in embed file

In [ ]:
from spyce.utils import load_specifications
from spyce.kmerMatrix import KMerCollection

# Create output path
data_path = "results/sPYce/"
Path(data_path).mkdir(exist_ok=True, parents=True)

# Then create embed file with paths to inputs

# Load embed file
setup_file_path = os.path.join(data_path, "spyce.embed")
setup_dict = load_specifications(setup_file_path)
setup_dict

In [ ]:
verbosity = 2
# Set k=4 for the example. Lower k are more efficient but might not keep all species specific differences.
# Values 4 <= k <= 7 are normally a good tradeoff between efficiency and accuracy.
k = 6
n_jobs = 48
# You can always normalize later. We recommend to save an unnormalized file.
# Normalize with the desired function once KMer object is loaded
normalization = "none"
# Highly recommended to remove KMers over regions that contain `N` sequences
n_policy = "remove"
# Keep repetitive sequences
mask_rep = False
# Account for GC-content bias in sequence. Peaks are sampled with replancement to match GC distribution
# over entire peak list
correct_gc = True
# We want that the forward and the reverse complement kmer aren't treated the same.
# Therefore `ACGC` has a different entry to `GCGT`. Note that there might be species specific diffences
# only due to the fact that the annotated strands are different. When `equalize_counter=False`,
# you can perform a kmer transposition, i.e. entries in `ACGC` are moved to `GCGT` and vice versa.
equalize_counter=False

kmer_collect = KMerCollection(
    setup_dict=setup_dict,
    k=k,
    verbosity=verbosity,
    n_jobs=n_jobs,
    normalization=normalization,
    correct_gc=correct_gc,
    n_policy=n_policy,
    mask_rep=mask_rep,
    equalize_counter=equalize_counter
)

save_prefix = f"k{k}"
save_path = "%s/%skmer_collection_obj.pkl" % (data_path, save_prefix)
kmer_collect.save(path=save_path, save_prefix=save_prefix)


# Normalization

In [ ]:
import os
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex, Normalize
from matplotlib.lines import Line2D
from matplotlib.cm import ScalarMappable
from spyce.kmerMatrix import KMerCollection
from spyce.plotting import plot_umap, plot_dr  # load plotting functions
from spyce.normalize import spyce_normalize  # decorator for creating your own normalization functions

In [ ]:
# set default values for UMAP and PCA
n_umap_neighbors = 30
min_dist = .01
n_pca_components = 50

Load KMer collection

In [ ]:
# Load kmer collection
k = 4
save_prefix = f"k{k}"

data_path = "results/sPYce/"
fig_dir = os.path.join("plots/sPYce/", save_prefix)
os.makedirs(fig_dir, exist_ok=True)

kmer_collect_path = "%s/%skmer_collection_obj.pkl" % (data_path, save_prefix)
kmer_collect = KMerCollection.load(kmer_collect_path)
kmer_collect

# KMerCollections are a wrapper for several KMerMatrix objects
kmer_collect.kmer_mat_list

# Get the shape of the object. First dimension is the number of cells, second is the k-mers.
# If you used k-mers of length 4, this would result in 4^4 = 256 combinations (or columns).
# For k=6, this results in 4^6 = 4096 combinations.
# The formula comes from the number of letters in the DNA code to the power of the k-mer length.
kmer_collect.shape

# This returns the number of cell that are held by the KMerCollection
len(kmer_collect)

# Every KMerCollection keeps the cell and sample information in a dedicated polars data table
# which you can access via kmer_mat_idc_df.
kmer_collect.kmer_mat_idc_df


Order of species cells in the collection

In [ ]:
# Get inidices for where mouse and nematostella data sets start
m_s = kmer_collect.kmer_mat_idc_df.filter(pl.col("species") == "mouse")["start index"].min()
m_e = kmer_collect.kmer_mat_idc_df.filter(pl.col("species") == "mouse")["end index"].max()

g_s = kmer_collect.kmer_mat_idc_df.filter(pl.col("species") == "nematostella_gastrula")["start index"].min()
g_e = kmer_collect.kmer_mat_idc_df.filter(pl.col("species") == "nematostella_gastrula")["end index"].max()

a_s = kmer_collect.kmer_mat_idc_df.filter(pl.col("species") == "nematostella_adult")["start index"].min()
a_e = kmer_collect.kmer_mat_idc_df.filter(pl.col("species") == "nematostella_adult")["end index"].max()


print("Mouse indices: %d-%d" % (m_s, m_e))
print("Nematostella gastrula indices: %d-%d" % (g_s, g_e))
print("Nematostella adult indices: %d-%d" % (a_s, a_e))

# Color for species
species_colors = {"mouse": "tab:blue", "nematostella_gastrula": "tab:orange", "nematostella_adult": "tab:green"}
species_vec = pd.concat([
    pd.Series(["mouse"] * (m_e - m_s)), 
    pd.Series(["nematostella_gastrula"] * (g_e - g_s)),
    pd.Series(["nematostella_adult"] * (a_e - a_s))
]).astype("category")
species_vec.index = np.arange(species_vec.shape[0])
species_c_vec = species_vec.map(species_colors)
species_c_vec.head()

In [ ]:
# Kmers histogram
from utils import plot_kmer_hist
top_kmers_hist, top_kmers = plot_kmer_hist(
    kmer_collect,
    species_colors=species_colors,
    top_n=40,
    save_path=os.path.join(fig_dir, "kmer_hist.pdf"))

Load cell type annotations

In [ ]:
# Load cell type annotations
mmus_fn = "/home/anamaria/cluster/aelek/proj/scATAC_nvec_v2/cross-species/mmus/adult/cell_metadata.txt"
mmus_an = pd.read_csv(mmus_fn, sep = "\t")

nvec_fn = "/home/anamaria/cluster/aelek/proj/scATAC_nvec_v2/Nematostella_scATAC/results/Clustering/Annotation_Adult_Gastrula_SEACell.tsv"
nvec_an = pd.read_csv(nvec_fn, sep = "\t")

mmus_an.rename(columns={"cell_label": "cell_type"}, inplace = True)
mmus_an["species"] = "mouse"
mmus_an["cell"] = mmus_an["species"] + mmus_an["cell"]

nvec_an.rename(columns={"stage": "tissue", "color": "cell_type_color"}, inplace = True)
nvec_an["species"] = "nematostella_" + nvec_an["tissue"]
nvec_an["cell"] = nvec_an["species"] + nvec_an["cell"]

# Colors for mouse
for x in ["cell_type", "tissue"]:
    ct_cmap = plt.get_cmap("rainbow")
    n_total = mmus_an[x].unique().shape[0] - 1
    colors = {
        ct: to_hex(ct_cmap(i_ct / float(n_total)))
        for i_ct, ct in enumerate(np.sort(mmus_an[x].unique()))
    }
    mmus_an[f"{x}_color"] = mmus_an[x].map(colors)

# Colors for Nematostella
stage_colors = {"adult": "#FFFF00", "gastrula": "#9F2B68"}
nvec_an["tissue_color"] = nvec_an["tissue"].map(stage_colors)

# Combine annotations
cell_annotation_df = pd.concat(
    [nvec_an[["cell", "cell_type", "cell_type_color", "tissue", "tissue_color", "species"]],
     mmus_an[["cell", "cell_type", "cell_type_color", "tissue", "tissue_color", "species"]]])
cell_annotation_df.set_index("cell", inplace=True)

# Subset annotations by KMerCollection
cell_annotation_df = cell_annotation_df[cell_annotation_df.index.isin(kmer_collect.barcodes)]

# Subset KMerCollection by cell annotations
kmer_collect = kmer_collect[cell_annotation_df.index]

# Order cell annotations by KMerCollection barcodes
cell_annotation_df = cell_annotation_df.loc[kmer_collect.barcodes]

# Save annotations
cell_annotation_df.to_csv(os.path.join(data_path, f"{save_prefix}kmer_cell_annotations.tsv"), sep="\t")

# Save filtered KMerCollection
save_path = "%s/%skmer_collection_filt_obj.pkl" % (data_path, save_prefix)
kmer_collect.save(path=save_path, save_prefix=save_prefix)

# Get inidices for mouse and nematostella data sets in new filtered KMerCollection
m_s = kmer_collect.kmer_mat_idc_df.filter(pl.col("species") == "mouse")["start index"].min()
m_e = kmer_collect.kmer_mat_idc_df.filter(pl.col("species") == "mouse")["end index"].max()

g_s = kmer_collect.kmer_mat_idc_df.filter(pl.col("species") == "nematostella_gastrula")["start index"].min()
g_e = kmer_collect.kmer_mat_idc_df.filter(pl.col("species") == "nematostella_gastrula")["end index"].max()

a_s = kmer_collect.kmer_mat_idc_df.filter(pl.col("species") == "nematostella_adult")["start index"].min()
a_e = kmer_collect.kmer_mat_idc_df.filter(pl.col("species") == "nematostella_adult")["end index"].max()

species_vec = pd.concat([
    pd.Series(["mouse"] * (m_e - m_s)), 
    pd.Series(["nematostella_gastrula"] * (g_e - g_s)),
    pd.Series(["nematostella_adult"] * (a_e - a_s))
]).astype("category")
species_vec.index = np.arange(species_vec.shape[0])
species_c_vec = species_vec.map(species_colors)
species_c_vec.head()

Cell type colors

In [ ]:
# Create a mapping from cell_type to color
ct_colors = cell_annotation_df[["cell_type", "cell_type_color"]].drop_duplicates().set_index("cell_type")["cell_type_color"].to_dict()

# Get the cell_type vector
ct_vec = cell_annotation_df["cell_type"]
ct_vec.index = np.arange(ct_vec.shape[0])

# Map to colors
ct_c_vec = ct_vec.map(ct_colors)
ct_c_vec.head()

Tissue colors

In [ ]:
# Create a mapping from cell_type to color
ts_colors = cell_annotation_df[["tissue", "tissue_color"]].drop_duplicates().set_index("tissue")["tissue_color"].to_dict()

# Get the cell_type vector
ts_vec = cell_annotation_df["tissue"]
ts_vec.index = np.arange(ts_vec.shape[0])

# Map to colors
ts_c_vec = ts_vec.map(ts_colors)
ts_c_vec.head()

For any downstream analysis, we should first reduce the dimensionality of the data set.

In [ ]:
# Dimensionality reduction of Kmer using PCA.
# This returns explained variance ratio.
kmer_collect.reduce_dimensionality(
    algorithm="pca",
    save_name="pca",
    n_pca_components=n_pca_components
)

Plot PCA with respect to the species.

In [ ]:
# Color wrt species
fig, ax = plot_dr(  # Dimensionality reduction plotting
    kmer_collect,
    ck=species_c_vec, # use the species color vector we created above
    cmap=None,
    randomize=True,
    title="PCA Species (no norm)",
)
handles = [
    Line2D(
        [0], [0], marker="o", color="w",
        markerfacecolor=c, markersize=5, label=s
    )
    for s, c in species_colors.items()
]
fig.legend(handles=handles, loc=7)
fig.tight_layout()
fig.subplots_adjust(right=0.7)
plt.savefig(os.path.join(fig_dir, "pca.pdf"))
plt.show()

The largest variance comes from the number of k-mers per cell.

In [ ]:
# Color wrt total number of KMers found per cell
fig, ax, scat = plot_dr(  # Dimensionality reduction plotting
    kmer_collect,
    ck=kmer_collect.kmer_hist.sum(axis=1),  # Color with respect to the number of k-mers found per cell
    cmap="jet",  # Use the jet cmap
    cmin=kmer_collect.kmer_hist.sum(axis=1).min(),  # Minimum value is the minimum number of k-mers
    cmax=kmer_collect.kmer_hist.sum(axis=1).max(),  # Maximum value is the maximum number of k-mers
    randomize=True,
    return_scat=True,
    title="PCA Number of found Kmers (no norm)",
)

fig.colorbar(
    ScalarMappable(
        norm=Normalize(
            kmer_collect.kmer_hist.sum(axis=1).min(),
            kmer_collect.kmer_hist.sum(axis=1).max()
        ),
        cmap="jet"), ax=ax, orientation="vertical"
)
fig.tight_layout()
fig.subplots_adjust(right=0.86)
plt.savefig(os.path.join(fig_dir, "pca.kmers.pdf"))
plt.show()

## Unit-Sum Normalization

In [ ]:
# Load kmer collection
k = 6
save_prefix = f"k{k}"

data_path = "results/sPYce/"

kmer_collect_path = "%s/%skmer_collection_filt_obj.pkl" % (data_path, save_prefix)
kmer_collect = KMerCollection.load(kmer_collect_path)
kmer_collect

# Normalize
kmer_collect.set_normalization("sum_norm")  # set normalization
kmer_collect

In [ ]:
# Kmers histogram
from utils import plot_kmer_hist
top_kmers_norm_hist, top_kmers_norm = plot_kmer_hist(
    kmer_collect,
    species_colors=species_colors, 
    kmer_list=top_kmers, 
    save_path=os.path.join(fig_dir, "kmer_hist.norm.pdf"))

In [ ]:
# PCA
kmer_collect.reduce_dimensionality(
    algorithm="pca",
    save_name="pca",
    n_pca_components=n_pca_components
)

In [ ]:
# Color wrt species
fig, ax = plot_dr(
    kmer_collect,
    ck=species_c_vec,
    cmap=None,
    randomize=True,
    title="PCA Species (norm-sum)",
)
handles = [
    Line2D(
        [0], [0], marker="o", color="w",
        markerfacecolor=c, markersize=5, label=s
    )
    for s, c in species_colors.items()
]
fig.legend(handles=handles, loc=7)
fig.tight_layout()
fig.subplots_adjust(right=0.7)
plt.savefig(os.path.join(fig_dir, "pca.norm.pdf"))
plt.show()

In [ ]:
# Calculate UMAP
kmer_collect.umap(
    dr_name="pca",  # Use the dimensionality reduction that is saved under the name pca,
    save_name="umap",  # This is also the default name, but set explicitly to explain function below
    n_neighbors=n_umap_neighbors,
    min_dist=min_dist
)

In [ ]:
# Color wrt species, tissues, cell types
vars_list = ["species", "tissue", "cell_type"]
cols_list = [species_colors, ts_colors, ct_colors]
cmap_list = [species_c_vec, ts_c_vec, ct_c_vec]

for x, y, z in zip(vars_list, cols_list, cmap_list):
    print(x)
    fig, ax = plot_umap(
        kmer_collect,
        ck=z.astype("string"),
        randomize=True,
        title=f"{x} (norm-sum)",
        cmap=None
    )
    handles = [
        Line2D(
            [0], [0], marker="o", color="w",
            markerfacecolor=c, markersize=5, label=s
        )
        for s, c in y.items()
    ]
    fig.legend(handles=handles, loc=7)
    fig.tight_layout()
    fig.subplots_adjust(right=0.7)
    plt.savefig(os.path.join(fig_dir, f"umap.norm.{x}.pdf"))

Unit-sum normalization successfully removed the count-based bias. However, the species still separate. Note that it is likely that each species has its own regulatory mechanisms, although the transcription factors used for a specific cell are conserved. Due to the presence of the species-specific processes, the unit-sum normalization gives different weights to the k-mers that should be conserved between the sepecies.

## Centered Unit-Sum Normalization

In [ ]:
# Load kmer collection
k = 6
save_prefix = f"k{k}"

data_path = "results/sPYce/"

kmer_collect_path = "%s/%skmer_collection_filt_obj.pkl" % (data_path, save_prefix)
kmer_collect = KMerCollection.load(kmer_collect_path)
kmer_collect

# Normalize
kmer_collect.set_normalization("centered_sum") 
kmer_collect

In [ ]:
# Kmers histogram
from utils import plot_kmer_hist
top_kmers_norm_cent_hist, top_kmers_norm_cent = plot_kmer_hist(
    kmer_collect,
    species_colors=species_colors, 
    kmer_list=top_kmers, 
    save_path=os.path.join(fig_dir, "kmer_hist.norm.cent.pdf"))

In [ ]:
# Perform PCA
kmer_collect.reduce_dimensionality(
    algorithm="pca",
    save_name="pca",
    n_pca_components=n_pca_components
)

In [ ]:
# Color wrt species
fig, ax = plot_dr(
    kmer_collect,
    ck=species_c_vec,
    dr_key="pca",
    cmap=None,
    randomize=True,
    title="PCA Species (centered-sum)",
)
handles = [
    Line2D(
        [0], [0], marker="o", color="w",
        markerfacecolor=c, markersize=5, label=s
    )
    for s, c in species_colors.items()
]
fig.legend(handles=handles, loc=7)
fig.tight_layout()
fig.subplots_adjust(right=0.7)
plt.savefig(os.path.join(fig_dir, "pca.norm.cent.pdf"))
plt.show()

In [ ]:
# Calculate UMAP
kmer_collect.umap(
    dr_name="pca",
    n_neighbors=n_umap_neighbors,
    min_dist=min_dist
)

In [ ]:
# Color wrt species, tissues, cell types
vars_list = ["species", "tissue", "cell_type"]
cols_list = [species_colors, ts_colors, ct_colors]
cmap_list = [species_c_vec, ts_c_vec, ct_c_vec]

for x, y, z in zip(vars_list, cols_list, cmap_list):
    print(x)
    fig, ax = plot_umap(
        kmer_collect,
        ck=z.astype("string"),
        randomize=True,
        title=f"{x} (centered-sum)",
        cmap=None
    )
    handles = [
        Line2D(
            [0], [0], marker="o", color="w",
            markerfacecolor=c, markersize=5, label=s
        )
        for s, c in y.items()
    ]
    fig.legend(handles=handles, loc=7)
    fig.tight_layout()
    fig.subplots_adjust(right=0.7)
    plt.savefig(os.path.join(fig_dir, f"umap.norm.cent.{x}.pdf"))

# Clustering

In [ ]:
import os
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex, Normalize
from matplotlib.lines import Line2D
from matplotlib.cm import ScalarMappable
from spyce.kmerMatrix import KMerCollection
from spyce.plotting import plot_umap, plot_dr  # load plotting functions
from spyce.normalize import spyce_normalize  # decorator for creating your own normalization functions

In [ ]:
# set default values for UMAP and PCA
n_umap_neighbors = 30
min_dist = .01
n_pca_components = 50

In [ ]:
# Load kmer collection
k = 4
save_prefix = f"k{k}"

data_path = "results/sPYce/"

fig_dir = os.path.join("plots/sPYce/", save_prefix)
os.makedirs(fig_dir, exist_ok=True)

kmer_collect_path = "%s/%skmer_collection_filt_obj.pkl" % (data_path, save_prefix)
kmer_collect = KMerCollection.load(kmer_collect_path)
kmer_collect

# Normalize
kmer_collect.set_normalization("centered_sum") 

# Perform PCA
kmer_collect.reduce_dimensionality(
    algorithm="pca",
    save_name="pca",
    n_pca_components=n_pca_components
)


In [ ]:
# Load annotation metadata
annot_path = os.path.join(data_path, f"{save_prefix}kmer_cell_annotations.tsv")
cell_annotation_df = pd.read_csv(annot_path, sep="\t", index_col=0)

# Create a mapping from cell_type to color
ct_colors = cell_annotation_df[["cell_type", "cell_type_color"]].drop_duplicates().set_index("cell_type")["cell_type_color"].to_dict()
ct_vec = cell_annotation_df["cell_type"]
ct_vec.index = np.arange(ct_vec.shape[0])
ct_c_vec = ct_vec.map(ct_colors)
ct_c_vec.head()

# Create a mapping from cell_type to color
ts_colors = cell_annotation_df[["tissue", "tissue_color"]].drop_duplicates().set_index("tissue")["tissue_color"].to_dict()
ts_vec = cell_annotation_df["tissue"]
ts_vec.index = np.arange(ts_vec.shape[0])
ts_c_vec = ts_vec.map(ts_colors)
ts_c_vec.head()

# Create combined mapping by using mouse 
cell_annotation_df[['annotation']] = 

# Get inidices for where mouse and nematostella data sets start
m_s = kmer_collect.kmer_mat_idc_df.filter(pl.col("species") == "mouse")["start index"].min()
m_e = kmer_collect.kmer_mat_idc_df.filter(pl.col("species") == "mouse")["end index"].max()
g_s = kmer_collect.kmer_mat_idc_df.filter(pl.col("species") == "nematostella_gastrula")["start index"].min()
g_e = kmer_collect.kmer_mat_idc_df.filter(pl.col("species") == "nematostella_gastrula")["end index"].max()
a_s = kmer_collect.kmer_mat_idc_df.filter(pl.col("species") == "nematostella_adult")["start index"].min()
a_e = kmer_collect.kmer_mat_idc_df.filter(pl.col("species") == "nematostella_adult")["end index"].max()

# Color for species
species_colors = {"mouse": "tab:blue", "nematostella_gastrula": "tab:orange", "nematostella_adult": "tab:green"}
species_vec = pd.concat([
    pd.Series(["mouse"] * (m_e - m_s)), 
    pd.Series(["nematostella_gastrula"] * (g_e - g_s)),
    pd.Series(["nematostella_adult"] * (a_e - a_s))
]).astype("category")
species_vec.index = np.arange(species_vec.shape[0])
species_c_vec = species_vec.map(species_colors)
species_c_vec.head()

## Batch correction

Remove non-linear species effects that can occur due to unequal cell type distribution.

In [ ]:
# Note that you need to have normalized and dimensionality reduced the data set first.
# This does not work when running directly on the k-mer histograms.
kmer_collect.remove_species_effect(
    batch_vec=species_vec,  # indicate along which values a batch/species effect can occur
    dr_key="pca",  # Use the dimensionality reduction saved under the pca key
    save_name="harmony_pca",  # save the Harmony corrected PCs under harmony_pca
    algorithm="harmony"  # use Harmony instead of ICP
)


In [ ]:
# Calculate UMAP
kmer_collect.umap(
    dr_name="harmony_pca",
    n_neighbors=15,
    min_dist=0.01,
    spread=1.5
)

# Color wrt species, tissues, cell types
vars_list = ["species", "tissue", "cell_type"]
cols_list = [species_colors, ts_colors, ct_colors]
cmap_list = [species_c_vec, ts_c_vec, ct_c_vec]

for x, y, z in zip(vars_list, cols_list, cmap_list):
    print(x)
    fig, ax = plot_umap(
        kmer_collect,
        ck=z.astype("string"),
        randomize=True,
        title=f"{x} (centered-sum)",
        cmap=None
    )
    handles = [
        Line2D(
            [0], [0], marker="o", color="w",
            markerfacecolor=c, markersize=5, label=s
        )
        for s, c in y.items()
    ]
    fig.legend(handles=handles, loc=7)
    fig.tight_layout()
    fig.subplots_adjust(right=0.7)
    plt.savefig(os.path.join(fig_dir, f"umap.norm.cent.harmony.{x}.pdf"))
